# Modeling and Evaluation

The dataset has now been cleaned, explored, and enriched through feature engineering and preprocessing. In the previous notebooks, we handled missing values, investigated the relationships between passenger characteristics and survival, created new features motivated by domain knowledge, and encoded categorical variables into numerical representations suitable for machine learning.

In this notebook, we transition from data preparation to predictive modeling. Our objective is to train and evaluate a variety of machine learning algorithms capable of predicting passenger survival based on the available passenger information.

We begin by constructing the final modeling dataset using the feature registry developed during feature engineering. This registry provides a transparent and reproducible specification of the features included in each model.

The following sections will split the data into training and validation sets, train multiple classification models, compare their predictive performance using appropriate evaluation metrics, and analyze the strengths and limitations of each approach.

## Import Libraries

We begin by importing the libraries required for model development and evaluation. These include tools for data manipulation, dataset splitting, machine learning algorithms, performance metrics, and visualization of the results.

In [8]:
# Import libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

## Data Loading

We begin by loading the processed dataset and feature registry produced in the previous notebook. At this stage, all preprocessing, feature engineering, transformations, and encoding steps have already been completed, allowing us to focus exclusively on model development and evaluation.

The feature registry provides a centralized specification of every available feature and indicates which variables should be included in the machine learning models.

In [9]:
# Load processed dataset and feature registry
df_full = pd.read_parquet("../data/processed/02_data.parquet")
df_features = pd.read_parquet("../data/processed/02_features.parquet")

In [10]:
df_full.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Age_Child,Age_Teen,Age_YoungAdult,Age_Adult,Age_Senior
0,1,0.0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,...,False,False,True,False,False,False,False,True,False,False
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,...,False,False,False,True,False,False,False,False,True,False
2,3,1.0,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,...,False,True,False,False,False,False,False,True,False,False
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,...,False,False,False,True,False,False,False,True,False,False
4,5,0.0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,...,False,False,True,False,False,False,False,True,False,False


In [11]:
df_features.head()

,Name,Type,Category,Source,Description,UsedInModel
0,Pclass,numerical,original,Pclass,"Passenger ticket class (`1` = First, `2` = Sec...",True
1,Name,categorical,original,Name,Passenger's full name.,False
2,Sex,categorical,original,Sex,Passenger's sex (`male` or `female`).,True
3,Age,numerical,original,Age,Passenger's age in years. Fractional values re...,True
4,SibSp,numerical,original,SibSp,Number of siblings and spouses aboard the Tita...,True


## Train/Test Split

The original Kaggle Titanic dataset provides separate training and test sets. Throughout the preprocessing pipeline, this distinction was preserved using the `IsTrainSet` indicator. We now use this flag to reconstruct the original split, ensuring that models are trained only on labeled data while retaining the unlabeled test set for generating the final competition predictions.

In [12]:
feature_names = df_features.loc[
    df_features["UsedInModel"],
    "Name"
]

X_train = df_full.loc[df_full["IsTrainSet"], feature_names]
y_train = df_full.loc[df_full["IsTrainSet"], "Survived"]

X_test = df_full.loc[~df_full["IsTrainSet"], feature_names]